In [12]:
import pandas as pd
from pathlib import Path
import yaml

def load_config(path):
    """
    Load event config file and data.

    Returns: Config `cfg` and DataFrame `df`.
    """
    config_file = path
    cfg = yaml.safe_load(Path(config_file).read_text(encoding='utf-8')).get('event', {})
    return cfg


In [29]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("input", type=str)
args = parser.parse_args()


config_file = args.input

cfg = load_config(config_file)
cfg

usage: ipykernel_launcher.py [-h] input
ipykernel_launcher.py: error: unrecognized arguments: -f


SystemExit: 2

/home/jupyter/miniconda3/envs/jupyter-lab/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
df = pd.read_csv(cfg['input_file'])

df.columns

# Output, for each 'domain' (url): lists of Actor1Code, Actor1Name, Actor1EthnicCode, <Actor2...>
# EventBaseCode, EventRootCode, GoldsteinScale, AvgTone.




Index(['GlobalEventID', 'date', 'Year', 'Actor1Code', 'Actor1Name',
       'Actor1CountryCode', 'Actor1EthnicCode', 'Actor2Code', 'Actor2Name',
       'Actor2CountryCode', 'Actor2EthnicCode', 'IsRootEvent', 'EventCode',
       'EventBaseCode', 'EventRootCode', 'QuadClass', 'GoldsteinScale',
       'NumMentions', 'NumSources', 'NumArticles', 'AvgTone',
       'Actor1Geo_CountryCode', 'Actor2Geo_CountryCode', 'ActionGeo_Type',
       'ActionGeo_Fullname', 'ActionGeo_CountryCode', 'ActionGeo_ADM1Code',
       'ActionGeo_Lat', 'ActionGeo_Long', 'ActionGeo_FeatureID', 'DATEADDED',
       'SOURCEURL', 'domain', 'target', 'title', 'text', 'description',
       'datetime', 'first_para'],
      dtype='object')

In [25]:
# Group entries

g = df.groupby('SOURCEURL').agg({
    'Actor1Code': list,
    'Actor1Name': list,
    'Actor1EthnicCode': list,
    'Actor2Code': list,
    'Actor2Name': list,
    'EventCode': list,
    'EventRootCode': list,
    'EventBaseCode': list,
    'GoldsteinScale': list,
    'AvgTone': list,
})
g.head(5)

,Actor1Code,Actor1Name,Actor1EthnicCode,Actor2Code,Actor2Name,EventCode,EventRootCode,EventBaseCode,GoldsteinScale,AvgTone
SOURCEURL,,,,,,,,,,
HTTP://en.ce.cn/main/latest/202205/24/t20220524_37611553.shtml,[AFG],[AFGHANISTAN],[nan],[IMGMOSALQ],[OSAMA BIN LADEN],[190],[19],[190],[-10.0],[-8.31858407079642]
HTTP://en.ce.cn/main/latest/202206/11/t20220611_37745220.shtml,"[VEN, CUB, BLZGOV, BLZGOV, MED, MEX]","[VENEZUELA, CUBA, BELIZE, BELIZE, MEDIA, MEXICO]","[nan, nan, nan, nan, nan, nan]","[BLZGOV, BLZGOV, CUB, VEN, nan, SLV]","[BELIZE, BELIZE, CUBA, VENEZUELA, nan, EL SALV...","[40, 40, 40, 40, 10, 125]","[4, 4, 4, 4, 1, 12]","[40, 40, 40, 40, 10, 125]","[1.0, 1.0, 1.0, 1.0, 0.0, -5.0]","[-4.27350427350427, -4.27350427350427, -4.2735..."
http://7thspace.com/headlines/1839076/rthk__abortion_rights_groups_kick_off__summer_of_rage_.html,[HRI],[RIGHTS GROUP],[nan],[nan],[nan],[111],[11],[111],[-2.0],[-5.95611285266458]
http://7thspace.com/headlines/1847172/u_s__citizen_and_four_chinese_intelligence_officers_charged_with_spying_on_prominent_dissidents__human_rights_leaders__and_pro_democracy_activists.html,[CRM],[CRIMINAL],[nan],[nan],[nan],[114],[11],[114],[-2.0],[-5.29953917050691]
http://7thspace.com/headlines/1847173/statement_on_the_fbi_response_to_the_shooting_in_buffalo__new_york.html,[USA],[UNITED STATES],[nan],[nan],[nan],[18],[1],[18],[3.4],[-4.14364640883978]


In [28]:
config_file = Path(config_file)
output_dir = Path(f"output/{config_file.stem}")

g.reset_index().to_csv(output_dir.joinpath("agg_codes.csv"), index=None)